# Tutorial 4: Geometry, Validation, And Performance

This notebook connects three questions that arise after a first `sfincs_jax` run: which geometry input should I use, how do I validate the output, and how do I make CPU/GPU runs efficient without manually choosing solver internals?

## Geometry Choices

`sfincs_jax` supports analytic tokamak/stellarator geometries, Boozer-coordinate files, and VMEC `wout` geometry. For a VMEC workflow, the user-facing contract is simple: keep `geometryScheme = 5` in the namelist and pass the equilibrium through `--wout-path` or the namelist `equilibriumFile`. The code loads geometry once, builds the drift-kinetic operator on the requested flux surface, and writes enough metadata to audit the run.

In [ ]:
from pathlib import Path

repo = Path.cwd()
getting_started = repo / "examples" / "getting_started"
vmec_example = getting_started / "write_sfincs_output_vmec.py"
tokamak_example = getting_started / "write_sfincs_output_tokamak.py"
vmec_example, tokamak_example

## One Command Pattern For VMEC

For a real VMEC run, use the CLI pattern below. The same command works on CPU or GPU depending on the installed JAX backend. The default solver policy selects a validated route automatically; advanced users can still override the solver method for reproducibility studies.

In [ ]:
print("sfincs_jax /path/to/input.namelist --wout-path /path/to/wout.nc --out sfincsOutput.h5")
print("sfincs_jax --plot sfincsOutput.h5")

## Validation Workflow

Validation has two layers. Fast tests compare algebraic kernels, output schema, frozen SFINCS Fortran v3 references, and literature-anchored trends without running a full suite. Benchmark scripts then rerun selected CPU/GPU/Fortran cases at production resolution when a performance or publication claim needs fresh evidence.

In [ ]:
validation_scripts = [
    repo / "examples" / "parity" / "output_parity_vs_fortran_fixture.py",
    repo / "examples" / "publication_figures" / "generate_validation_dashboard.py",
    repo / "examples" / "publication_figures" / "generate_fortran_suite_benchmark_summary.py",
]
for script in validation_scripts:
    print(script.relative_to(repo), script.exists())

## Performance And Parallelism

Small examples are useful for learning, but reliable runtime comparisons need enough work per process or device. For research-scale sweeps, use warm timings, keep the same grid/resolution across SFINCS Fortran v3 and `sfincs_jax`, and compare CPU cold, CPU warm, GPU cold, and GPU warm results. Independent radii, electric-field points, and RHS columns are the most robust parallelism targets.

In [ ]:
performance_scripts = [
    repo / "examples" / "performance" / "benchmark_output_formats.py",
    repo / "examples" / "performance" / "benchmark_transport_l11_vs_fortran.py",
]
for script in performance_scripts:
    print(script.relative_to(repo), script.exists())

## Reading Solver Metadata

Every production output should be checked for convergence metadata before using fluxes, bootstrap current, or ambipolar roots in analysis. For RHSMode=1, inspect `linearSolverConverged`, `linearSolverResidualNorm`, `linearSolverResidualTarget`, and `linearSolverMethod`. For transport matrices, inspect the solver trace JSON and benchmark summary rows.

In [ ]:
important_output_fields = [
    "linearSolverMethod",
    "linearSolverConverged",
    "linearSolverResidualNorm",
    "linearSolverResidualTarget",
    "FSABjHatOverRootFSAB2",
]
important_output_fields

## What To Run Next

Use `examples/getting_started/` for minimal geometry scripts, `examples/parity/` for frozen SFINCS Fortran v3 checks, `examples/performance/` for CPU/GPU timing and memory studies, and `examples/vmec_jax_finite_beta/` for VMEC-to-bootstrap-current workflows. Keep heavy generated outputs outside the repository and store large reference data through the release-data cache.